# RunnablePassthrough : input ko bina change kiye aage pass karna yeah output mein return kar do..

Iska practical use?
Maan lo tumhare paas input hai aur tum parallel processing mein original input bhi preserve karna chahte ho.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough()

result = chain.invoke("Hello Bro!")

print(result) # output : Hello Bro!    input aur output same hain!

iska Actual use samjh-RunnableParallel ke saath karna hai. Tumhare paas 2 tasks hai, ek task input ko process karega aur dusra task original input ko preserve karega. Tum RunnablePassthrough ka use karke original input ko preserve kar sakte ho aur RunnableParallel ke through dono tasks ko simultaneously execute kar sakte ho.


In [ ]:
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda
)

chain = RunnableParallel(
    original=RunnablePassthrough(),

    greeting=RunnableLambda(
        lambda data: f"Hello {data['name']}!"
    )
)

result = chain.invoke({
    "name": "Rahul",
    "topic": "Python"
})

print(result)


#output
# {
#     "original": {
#         "name": "Rahul",
#         "topic": "Python"
#     },
#     "greeting": "Hello Rahul!"
# }

# 5. RunnableSequence
Multiple Runnables ko ek sequence mein execute karna.

Ye wahi concept hai jo hum | operator se kar rahe the.

In [ ]:
chain = (
    prompt_template
    | llm
    | StrOutputParser()
    | RunnableLambda(lambda text: text.upper())
)

result = chain.invoke({"topic": "Python"})

# Message History : RunnableWithMessageHistory : e LangChain mein chatbot ki conversation history maintain karne ke liye use hota hai.


Maan lo user ne poocha:
User: My name is Rahul.
AI: Nice to meet you Rahul!

Ab user poochta hai:

User: What is my name?

Agar previous conversation model ko nahi bheji, toh usko Rahul yaad nahi hoga.

Yahan use hota hai Message History.

LangChain mein ek approach hai RunnableWithMessageHistory.


Conceptual flow:
User message
     |
Load previous messages

LLM receives conversation history + new message

Generate response

Save conversation history

Problem: Second question ke time AI ko first conversation yaad nahi hai.

Kyun? Normally har llm.invoke() call mein model ko sirf wahi input milta hai jo tum bhejte ho. Previous messages automatically yaad nahi rehte.

Solution: Message History
Hum previous messages ko store karenge aur next request ke saath model ko bhejenge.

In [ ]:
Conceptual flow:
User message
      |
Load previous messages
      |
LLM receives conversation history + new message
      |
Generate response
       |
Save conversation history

RunnableWithMessageHistory kya karta hai?
=> Ye existing LangChain chain ke around ek wrapper ki tarah kaam karta hai.


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_core.chat_history import InMemoryChatMessageHistory

from langchain_core.runnables.history import RunnableWithMessageHistory



# A. MessagesPlaceholder
# MessagesPlaceholder(variable_name="history")



# Ye prompt mein ek jagah reserve karta hai jahan previous conversation messages insert honge.

# B. Store
# store = {}



# Ye dictionary hai jisme different users ki conversation history store hogi.

# C. get_session_history()
# def get_session_history(session_id):    if session_id not in store:        store[session_id] = InMemoryChatMessageHistory()    return store[session_id]



# Agar user pehli baar aaya toh uski history create karo. Agar pehle se hai toh existing history return karo.

# D. RunnableWithMessageHistory
# chatbot = RunnableWithMessageHistory(    chain,    get_session_history,    input_messages_key="input",    history_messages_key="history")

# Ye normal chain ko history-aware chain bana deta hai.


# 1. Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini")


# 2. Create prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])


# 3. Create normal chain
chain = prompt | llm


# 4. Store conversation histories
store = {}

def get_session_history(session_id: str):

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]


# 5. Add message history to chain
chatbot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)


# 6. First conversation
response1 = chatbot.invoke(
    {"input": "Hi, my name is Rahul."},
    config={"configurable": {"session_id": "user_1"}}
)

print(response1.content)


# 7. Second conversation
response2 = chatbot.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user_1"}}
)

print(response2.content)